Run this cell to install genexp into your environment if you haven't already done so (e.g. on Colab)

In [ ]:
!pip install https://github.com/Komod0D/flow-expansion.git

In [ ]:
from matplotlib import pyplot as plt
from tqdm.notebook import tqdm
import torch
import copy

import diffusiongym
from omegaconf import OmegaConf
from genexp.trainers.genexp import FlowExpansionTrainer

In [ ]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

env = diffusiongym.make(
    base_model="1d/trimodal_gmm",
    reward="1d/binary",
    discretization_steps=50,
    device=device,
)

In [ ]:
import torch.distributions as dist
import numpy as np

# True GMM density for reference
p_true = dist.MixtureSameFamily(
    dist.Categorical(torch.tensor([0.6, 0.2, 0.2])),
    dist.Normal(torch.tensor([0.0, 1.0, -1.0]), torch.tensor([0.4, 0.2, 0.2])),
)

with torch.no_grad():
    samples = env.sample(5000, pbar=True).sample.data.cpu().squeeze()

x_range = torch.linspace(-3, 4, 500)
true_density = p_true.log_prob(x_range).exp().numpy()

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(samples.numpy(), bins=80, density=True, alpha=0.6, label='model samples', color='steelblue')
ax.plot(x_range.numpy(), true_density, 'r-', lw=2, label='true GMM density')
ax.set_xlabel('x')
ax.set_ylabel('density')
ax.set_title('Trimodal GMM base model: large mode at 0, smaller modes at ±1')
ax.legend()
plt.tight_layout()
plt.show()

Create environment — this trains the base model automatically

Visualize pre-trained model density (takes a few minutes to sample enough poins)

In [ ]:
base_samples = env.sample(5000, pbar=True).sample.data.cpu().squeeze()

plt.hist(base_samples.numpy(), bins=50, density=True, alpha=0.5, label='pre-trained model density')
plt.legend()
plt.show()

Initialize Flow Expansion trainer

In [ ]:
config = OmegaConf.create({
    'gamma': 1.0,
    'eta': 1.0,
    'beta': 0.0,
    'epsilon': 0.01,
    'traj': True,
    'lmbda': 'const',
    'num_md_iterations': 3,
    'adjoint_matching': {
        'lr': 1e-4,
        'batch_size': 128,
        'num_iterations': 2,
        'finetune_steps': 50,
        'sampling': {
            'num_samples': 512,
        },
    },
})

model = copy.deepcopy(env.base_model)
base_model = copy.deepcopy(env.base_model)

fe_trainer = FlowExpansionTrainer(config, env, model, base_model, device=device)

Main Flow Expansion loop

In [ ]:
for k in tqdm(range(config.num_md_iterations)):
    fe_trainer.expand()

    for i in range(config.adjoint_matching.num_iterations):
        dataset = fe_trainer.generate_dataset()
        fe_trainer.finetune(dataset, steps=config.adjoint_matching.finetune_steps)

    if fe_trainer.grad_constraint is not None and fe_trainer.eta_coeff > 0.:
        fe_trainer.project()

        for i in range(config.adjoint_matching.num_iterations):
            dataset = fe_trainer.generate_dataset()
            fe_trainer.finetune(dataset, steps=config.adjoint_matching.finetune_steps)

    fe_trainer.update_base_model()

Visualizing fine-tuned density (takes a few minutes to sample enough points)

In [ ]:
env.base_model = fe_trainer.fine_model
fine_samples = env.sample(5000, pbar=True).sample.data.cpu().squeeze()
env.base_model = fe_trainer.base_base_model

In [ ]:
plt.hist(base_samples.numpy(), bins=50, density=True, alpha=0.5, label='pre-trained model density', color='orange')
plt.hist(fine_samples.numpy(), bins=50, density=True, alpha=0.5, label='fine-tuned model density', color='green')
plt.legend()
plt.show()